# 1. General

In [ ]:
import os, pandas as pd
from injustos.utils.config_loader import load_hparams
from injustos.utils.misc import mask_ids
from injustos.trainer import train_one_from_splits
from injustos.reports.plots import plot_losses, plot_f1 
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [ ]:
CSV_PATH = "local_database/ToS_100/ToS_100.csv"
KB_DIR   = "local_database/KB"
DIST_CFG = "configs/distributed_model_config.json"
FOLD_DIR = "cv_test/torch/fold_1"         
MAX_LEN  = 128
BATCH    = 32                        
SEED     = 42
TRAIN_FRAC = 0.70   
VAL_FRAC   = 0.15
TEST_FRAC  = 0.15 

In [ ]:
os.makedirs(FOLD_DIR, exist_ok=True)
hparams = load_hparams(DIST_CFG)
df = pd.read_csv(CSV_PATH)

if "label" in df.columns:
    y = df["label"].astype(int).values

X_Tv, X_t, y_Tv, y_t = train_test_split(
    df, y, test_size = TEST_FRAC, random_state = SEED, shuffle = True, stratify = y
)

X_T, X_v, y_T, y_v = train_test_split(
    X_Tv, y_Tv, test_size = VAL_FRAC / (VAL_FRAC + TRAIN_FRAC), random_state = SEED, shuffle = True, stratify = y_Tv
)

tr_csv = os.path.join(FOLD_DIR, "train.csv")
va_csv = os.path.join(FOLD_DIR, "val.csv")
te_csv = os.path.join(FOLD_DIR, "test.csv")

os.makedirs(FOLD_DIR, exist_ok=True)
X_T.to_csv(tr_csv, index=False)
X_v.to_csv(va_csv, index=False)
X_t.to_csv(te_csv, index=False)

X_T.shape, X_v.shape, X_t.shape

# 2. Entrenamiento

In [ ]:
best_ckpt, metrics = train_one_from_splits(
    train_csv=tr_csv,
    val_csv=va_csv,
    test_csv=te_csv,
    hparams=hparams,
    out_dir=FOLD_DIR,
    kb_dir=KB_DIR,
    max_len=MAX_LEN,
    batch_size=BATCH
)
print("best_ckpt:", best_ckpt)
print("metrics :", metrics)

In [ ]:
plot_losses("cv_test/torch/fold_1/logs/version_0/metrics.csv", smooth=1, show=True, save_path=os.path.join("cv_test/torch/fold_1/logs/version_0", "loss.png"))
plot_f1("cv_test/torch/fold_1/logs/version_0/metrics.csv",     smooth=1, show=True, save_path=os.path.join("cv_test/torch/fold_1/logs/version_0", "f1.png"))

# 3. Gráficos

In [ ]:
import torch
from sklearn.metrics import classification_report
from injustos.utils.config_loader import load_model_and_tokenizer
from injustos.dataset import ToS, make_dataloaders

lit, tok, kb_struct = load_model_and_tokenizer(
    ckpt_path=best_ckpt,
    max_len=MAX_LEN,
    kb_dir=KB_DIR,
    map_location="cuda",  
)
lit.eval()
ds_test = ToS(pd.read_csv(te_csv), tok, MAX_LEN)
_, _, te_loader = make_dataloaders(ds_test, None, ds_test, batch_size=BATCH)

all_probs, all_preds, all_gold = [], [], []
device = next(lit.parameters()).device

with torch.no_grad():
    for batch in te_loader:
        x = batch["input_ids"].to(device, non_blocking=True)
        logits, *_ = lit(x)
        probs = logits.sigmoid().cpu()
        preds = (probs > 0.5).to(torch.int)
        all_probs.append(probs)
        all_preds.append(preds)
        all_gold.append(batch["labels_multi"].cpu())

import numpy as np
P = torch.cat(all_preds).numpy()
G = torch.cat(all_gold).numpy()

print("=== Multi-label (A,CH,CR,LTD,TER) ===")
print(classification_report(G, P, target_names=["A","CH","CR","LTD","TER"], zero_division=0))

Pg = (P.max(axis=1) > 0).astype(int)
Gg = (G.max(axis=1) > 0).astype(int)
print("=== General (OR derivado) ===")
print(classification_report(Gg, Pg, target_names=["fair","unfair"], zero_division=0))